# NLP Text Classification

Цель проекта — построить модель классификации новостей по тексту.

На первом этапе:
- загрузим реальные текстовые данные;
- построим классический baseline;
- разберём, как текст превращается в числа;
- затем обучим первую NLP-модель на PyTorch.

In [3]:
from datasets import load_dataset

dataset = load_dataset("sh0416/ag_news")

print(dataset)

README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 33.7MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})


## 1. Структура данных

Каждая новость содержит:

- `title` — заголовок;
- `description` — текст новости;
- `label` — категория, которую должна предсказывать модель.

In [7]:
sample = dataset["train"][0]

print(sample)
print()
print(dataset["train"].features)

{'label': 3, 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}

{'label': Value('int64'), 'title': Value('string'), 'description': Value('string')}


## 2. Классы

Проверим, какие значения принимает `label`, и посмотрим по одному примеру новости из каждого класса.

In [12]:
labels = sorted(dataset["train"].unique("label"))

print(f"labels: {labels}")

examples = {label: [] for label in labels}

for row in dataset["train"]:
    label = row["label"]

    if len(examples[label]) < 2:
        examples[label].append(row)

    if all(len(rows) == 2 for rows in examples.values()):
        break

for label, rows in examples.items():
    print()
    print(f"label: {label}")
    for row in rows:
        print(f"- {row['title']}")

labels: [1, 2, 3, 4]

label: 1
- Venezuelans Vote Early in Referendum on Chavez Rule (Reuters)
- S.Koreans Clash with Police on Iraq Troop Dispatch (Reuters)

label: 2
- Phelps, Thorpe Advance in 200 Freestyle (AP)
- Reds Knock Padres Out of Wild-Card Lead (AP)

label: 3
- Wall St. Bears Claw Back Into the Black (Reuters)
- Carlyle Looks Toward Commercial Aerospace (Reuters)

label: 4
- 'Madden,' 'ESPN' Football Score in Different Ways (Reuters)
- Group to Propose New High-Speed Wireless Format (Reuters)


In [13]:
label_names = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

## 3. Подготовка текста

Объединим `title` и `description` в одно текстовое поле.

Так модель будет получать одновременно заголовок и описание новости.

In [14]:
def combine_text(row):
    return {
        "text": row["title"] + " " + row["description"]
    }

dataset = dataset.map(combine_text)

print(dataset["train"][0]["text"])

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


## 4. TF-IDF baseline

Модель не умеет работать со строками напрямую, поэтому сначала преобразуем тексты в числовые признаки.

Для первого baseline используем TF-IDF.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words="english"
)

X_train = vectorizer.fit_transform(dataset["train"]["text"])
X_test = vectorizer.transform(dataset["test"]["text"])

y_train = dataset["train"]["label"]
y_test = dataset["test"]["label"]

print(X_train.shape)
print(X_test.shape)

(120000, 20000)
(7600, 20000)


### Что находится внутри TF-IDF-вектора

Посмотрим, какие слова первой новости получили ненулевые TF-IDF-веса.

In [27]:
feature_names = vectorizer.get_feature_names_out()

first_vector = X_train[0]

indices = first_vector.nonzero()[1]
values = first_vector.data

for index, value in zip(indices, values):
    print(f"{feature_names[index]}: {value:.3f}")

wall: 0.384
st: 0.195
bears: 0.244
claw: 0.339
black: 0.218
reuters: 0.223
short: 0.206
sellers: 0.309
street: 0.187
dwindling: 0.318
band: 0.260
ultra: 0.289
seeing: 0.269
green: 0.213


## 5. Logistic Regression baseline

Используем TF-IDF-признаки для обучения простого классификатора.

Это будет baseline, с которым позже сравним нейросетевую модель.

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"accuracy: {accuracy:.3f}")
print()
print(classification_report(
    y_test,
    predictions,
    target_names=["World", "Sports", "Business", "Sci/Tech"]
))

accuracy: 0.915

              precision    recall  f1-score   support

       World       0.93      0.90      0.92      1900
      Sports       0.96      0.98      0.97      1900
    Business       0.88      0.88      0.88      1900
    Sci/Tech       0.89      0.89      0.89      1900

    accuracy                           0.92      7600
   macro avg       0.92      0.92      0.92      7600
weighted avg       0.92      0.92      0.92      7600



## 6. От TF-IDF к Embeddings

TF-IDF представляет весь текст одним большим разреженным вектором.

В нейросетевой модели каждому слову сначала присваивается числовой ID, а затем слой `Embedding` преобразует этот ID в обучаемый плотный вектор.

Эти векторы будут обучаться вместе с остальными параметрами модели.

## 7. Токенизация и vocabulary

Для нейросетевой модели преобразуем текст в последовательность токенов.

Затем построим vocabulary только по train-данным и присвоим каждому токену числовой ID.

Зарезервируем два специальных ID:

- `<PAD>` — заполнение коротких текстов;
- `<UNK>` — слова, которых нет в vocabulary.

In [31]:
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

token_counts = Counter()

for text in dataset["train"]["text"]:
    token_counts.update(tokenize(text))

max_vocab_size = 20000

most_common_tokens = token_counts.most_common(max_vocab_size - 2)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for token, _ in most_common_tokens:
    vocab[token] = len(vocab)


print(f"vocab size: {len(vocab)}")
print(list(vocab.items())[:20])

vocab size: 20000
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('to', 3), ('a', 4), ('of', 5), ('in', 6), ('and', 7), ('s', 8), ('on', 9), ('for', 10), ('that', 11), ('with', 12), ('as', 13), ('at', 14), ('its', 15), ('is', 16), ('new', 17), ('by', 18), ('it', 19)]


## 8. Преобразование текста в token IDs

Преобразуем каждый токен в его ID из vocabulary.

Все последовательности приведём к одинаковой длине:
- длинные тексты обрежем;
- короткие дополним `<PAD>`.

In [34]:
def encode_text(text, vocab, max_length=100):
    tokens = tokenize(text)

    token_ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    token_ids = token_ids[:max_length]

    padding_length = max_length - len(token_ids)

    token_ids += [vocab["<PAD>"]] * padding_length

    return torch.tensor(token_ids, dtype=torch.long)


sample_text = dataset["train"][0]["text"]
sample_ids = encode_text(sample_text, vocab)

print(sample_text)
print()
print(sample_ids)
print()
print(sample_ids.shape)

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

tensor([  431,   432,  1599, 13948,   106,    62,     2,   819,    21,    21,
          726,  7881,   431,   372,     8,  9852,  2808,     5,  5599,     1,
           40,  3923,   763,   326,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])

torch.Size([100])


In [39]:
tokens = tokenize(sample_text)[:100]

for token, token_id in zip(tokens, sample_ids):
    if token_id == vocab["<PAD>"]:
        break

    print(f"{token:10} → {token_id.item()}")

wall       → 431
st         → 432
bears      → 1599
claw       → 13948
back       → 106
into       → 62
the        → 2
black      → 819
reuters    → 21
reuters    → 21
short      → 726
sellers    → 7881
wall       → 431
street     → 372
s          → 8
dwindling  → 9852
band       → 2808
of         → 5
ultra      → 5599
cynics     → 1
are        → 40
seeing     → 3923
green      → 763
again      → 326


## 9. Dataset и DataLoader

Создадим PyTorch Dataset, который для каждой новости:

1. берёт текст;
2. преобразует его в последовательность token IDs;
3. возвращает token IDs и правильный класс.

Train-часть дополнительно разделим на train и validation.
Test пока не трогаем — он останется для финальной оценки модели.

In [40]:
from torch.utils.data import Dataset, DataLoader, random_split


class NewsDataset(Dataset):
    def __init__(self, hf_dataset, vocab):
        self.data = hf_dataset
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]

        token_ids = encode_text(
            row["text"],
            self.vocab
        )

        label = torch.tensor(
            row["label"] - 1,
            dtype=torch.long
        )

        return token_ids, label


full_train_dataset = NewsDataset(
    dataset["train"],
    vocab
)

train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

x_batch, y_batch = next(iter(train_loader))

print(f"x batch shape: {x_batch.shape}")
print(f"y batch shape: {y_batch.shape}")
print(f"first label: {y_batch[0]}")

x batch shape: torch.Size([64, 100])
y batch shape: torch.Size([64])
first label: 3


## 10. PyTorch-модель с Embedding

Модель будет:

1. преобразовывать token IDs в embedding-векторы;
2. усреднять embeddings токенов одной новости;
3. передавать полученный вектор в Linear-слой;
4. выдавать 4 числа — по одному для каждого класса.

In [41]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=num_classes
        )

    def forward(self, x):
        embedded = self.embedding(x)

        mask = (x != 0).unsqueeze(-1)

        summed = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)

        pooled = summed / lengths

        return self.linear(pooled)


model = TextClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    num_classes=4
)

output = model(x_batch)

print(f"output shape: {output.shape}")

output shape: torch.Size([64, 4])


## 11. Loss для многоклассовой классификации

Модель возвращает по одному числу для каждого класса.

Для обучения используем `CrossEntropyLoss`, который сравнивает эти значения с правильным номером класса.

In [42]:
loss_fn = nn.CrossEntropyLoss()

prediction = model(x_batch)

loss = loss_fn(
    prediction,
    y_batch
)

predicted_classes = prediction.argmax(dim=1)

print(f"prediction shape: {prediction.shape}")
print(f"first prediction: {prediction[0]}")
print(f"first predicted class: {predicted_classes[0]}")
print(f"first true class: {y_batch[0]}")
print(f"loss: {loss.item():.3f}")

prediction shape: torch.Size([64, 4])
first prediction: tensor([ 0.0094,  0.0098, -0.1656,  0.0546], grad_fn=<SelectBackward0>)
first predicted class: 3
first true class: 3
loss: 1.394
